In [6]:
import requests

def get_polymer_entity_ids(pdb_id):
    """Holt alle Polymer-Entity-IDs für eine gegebene PDB-ID"""
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id.lower()}"
    response = requests.get(url)
    if response.status_code != 200:
        raise RuntimeError(f"Fehler beim Abrufen der Polymer-Entity-IDs: {response.status_code}")
    data = response.json()
    return data['rcsb_entry_container_identifiers']['polymer_entity_ids']

def get_aa_sequence(pdb_id, entity_id):
    """Holt die Aminosäuresequenz für eine Polymer-Entity"""
    url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id.lower()}/{entity_id}"
    response = requests.get(url)
    if response.status_code != 200:
        raise RuntimeError(f"Fehler beim Abrufen der Sequenz für {pdb_id}_{entity_id}")
    data = response.json()
    seq = data.get('entity_poly', {}).get('pdbx_seq_one_letter_code_can')
    if seq:
        return f">{pdb_id}_{entity_id}\n{seq}"
    return None

def get_all_sequences(pdb_id):
    """Gibt alle Aminosäuresequenzen für eine PDB-ID zurück"""
    try:
        entity_ids = get_polymer_entity_ids(pdb_id)
        all_seqs = []
        for eid in entity_ids:
            seq = get_aa_sequence(pdb_id, eid)
            if seq:
                all_seqs.append(seq)
        return "\n\n".join(all_seqs) if all_seqs else "Keine Sequenzen gefunden."
    except Exception as e:
        return f"Fehler: {e}"

# Beispiel
pdb_id = "8WU1"
print(get_all_sequences(pdb_id))

>8WU1_1
MGDKGTRVFKKASPNGKLTVYLGKRDFVDHIDLVEPVDGVVLVDPEYLKERRVYVTLTCAFRYGREDLDVLGLTFRKDLFVANVQSFPPAPEDKKPLTRLQERLIKKLGEHAYPFTFEIPPNLPCSVTLQPGPEDTGKACGVDYEVKAFCAENLEEKIHKRNSVRLVIEKVQYAPERPGPQPTAETTRQFLMSDKPLHLEASLDKEIYYHGEPISVNVHVTNNTNKTVKKIKISVRQYADICLFNTAQYKCPVAMEEADDTVAPSSTFCKVYTLTPFLANNREKRGLALDGKLKHEDTNLASSTLLREGANREILGIIVSYKVKVKLVVSRGGLLGDLASSDVAVELPFTLMHPKPKEEPPHREVPEHETPVDTNLIELDTNDDDAAAEDFAR

>8WU1_2
EISEVQLVESGGGLVQPGGSLRLSCAASGFNVYSSSIHWVRQAPGKGLEWVASISSYYGYTYYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARSRQFWYSGLDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDKTHHHHHHHHH

>8WU1_3
SDIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQYKYVPVTFGQGTKVEIKRTVAAPSVFIFPPSDSQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC

>8WU1_4
MKSILDGLADTTFRTITTDLLYVGSNDIQYEDIKGDMASKLGYFPQKFPLTSFRGSPFQEKMTAGDNPQLVPADQVNITEFYNKSLSSFKENEENIQCGENFMDIECFMVLNPSQQ

Test, um zu schauen ob Code für API an einer pdb funktioniert 

*Protein Sequenzen*

Hier habe ich einen Code geschrieben, der für alle IDs aus df_sars_hum_cleaned_final.csv die aa Sequnz aus pdb in der Datei pdb_sequences_detailed.csv speichert. Es wird zusätzlich die Information gegeben ob es sich um die light/heavy chain des Abs oder um das Antigen handelt. Es wurde auch beachtet, dass alle pdb ID entities einzeln gezählt werden. 

In [8]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_polymer_entity_info(pdb_id):
    """Holt Infos zu allen Polymer-Entities einer PDB-ID"""
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id.lower()}"
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        entry = r.json()
        entity_ids = entry['rcsb_entry_container_identifiers']['polymer_entity_ids']
    except Exception:
        return []

    results = []
    for eid in entity_ids:
        try:
            url_entity = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id.lower()}/{eid}"
            r_entity = requests.get(url_entity, timeout=10)
            r_entity.raise_for_status()
            data = r_entity.json()
            seq = data.get('entity_poly', {}).get('pdbx_seq_one_letter_code_can', '')
            chains = ','.join(data.get('rcsb_polymer_entity_container_identifiers', {}).get('auth_asym_ids', []))
            organism = ','.join(data.get('rcsb_entity_source_organism', [{}])[0].get('scientific_name', '').split(';'))
            description = data.get('rcsb_polymer_entity', {}).get('pdbx_description', '')
            results.append({
                'pdb_id': pdb_id.upper(),
                'entity_id': eid,
                'chain_ids': chains,
                'sequence': seq,
                'organism': organism,
                'description': description
            })
        except Exception:
            continue
    return results

def process_pdb_ids(pdb_ids, max_workers=10):
    all_results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(get_polymer_entity_info, pid): pid for pid in pdb_ids}
        for future in as_completed(futures):
            try:
                result = future.result()
                all_results.extend(result)
            except Exception as e:
                print(f"[ERROR] {futures[future]} fehlgeschlagen: {e}")
    return pd.DataFrame(all_results)

def write_fasta(df, filename="pdb_sequences.fasta"):
    with open(filename, "w") as f:
        for _, row in df.iterrows():
            header = f">{row['pdb_id']}_{row['entity_id']}|{row['chain_ids']}|{row['organism']}"
            f.write(f"{header}\n{row['sequence']}\n\n")

# Beispiel-Verwendung
df_input = pd.read_csv("../data_cleanup/df_sars_hum_cleaned_final.csv", sep="\t")
unique_pdb_ids = df_input['pdb'].dropna().unique()
df_result = process_pdb_ids(unique_pdb_ids, max_workers=10)

# Speichern als CSV und FASTA
df_result.to_csv("pdb_sequences_detailed.csv", index=False)
write_fasta(df_result)
print("[DONE] Sequenzen gespeichert.")

FileNotFoundError: [Errno 2] No such file or directory: '../data_cleanup/df_sars_hum_cleaned_final.csv'

Als nächstes habe ich die .csv Datei in eine .fasta umgewandelt, um sie aufs BLASTEN vorzubereiten.

In [9]:
import pandas as pd

# CSV ohne Header einlesen und Spalten benennen
df = pd.read_csv("pdb_sequences_detailed.csv", header=None, usecols=[0, 1, 2, 3],
                 names=["pdb_id", "entity_id", "chain_id", "sequence"])

# FASTA-Datei schreiben
with open("pdb_sequences.fasta", "w") as fasta:
    for _, row in df.iterrows():
        header = f">{row['pdb_id']}_entity{row['entity_id']}_chain{row['chain_id']}"
        sequence = row['sequence'].replace(" ", "").replace("\n", "")
        fasta.write(f"{header}\n{sequence}\n")

FileNotFoundError: [Errno 2] No such file or directory: 'pdb_sequences_detailed.csv'

ich hab mal veruscht zu blasten, Befehle wurden im Terminal ausgeführt: 

makeblastdb -in documentation/pdb_sequences.fasta -dbtype prot -out pdbdb


blastp -query documentation/pdb_sequences.fasta -db pdbdb -out results.txt -outfmt 6